<h3>작은 인덱싱 비교 실험</h3>
<p><strong>개념</strong> 비교 실험은 질문과 문서를 고정하고 한 번에 하나의 조건만 바꿉니다. 여기서는 동일한 원문 하나에서 두 OpenAI 임베딩 모델의 검색 순위를 확인합니다.</p>

In [1]:
%pip install -q "python-dotenv==1.2.3" "openai==2.54.0" "langchain-text-splitters==1.1.2" "tiktoken==0.14.0"


Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from math import sqrt
import json

load_dotenv(find_dotenv(usecwd=True))
client = OpenAI()
text = "# 배송\n제주 배송은 기본 배송보다 이틀 더 걸립니다. 회원 등급은 매월 갱신합니다.\n\n# 환불\n환불은 수령 후 7일 이내 신청합니다."
query = "제주에는 배송이 얼마나 더 걸리나요?"
models = ["text-embedding-3-small", "text-embedding-3-large"]


In [3]:
recursive = RecursiveCharacterTextSplitter(
    chunk_size=45, chunk_overlap=8, keep_separator="end"
).split_text(text)
token_aware = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=24, chunk_overlap=4
).split_text(text)
structural = [block.strip() for block in text.split("\n\n") if block.strip()]
strategies = {"recursive": recursive, "token": token_aware, "structural": structural}
assert set(strategies) == {"recursive", "token", "structural"} and all(strategies.values())
print({name: len(chunks) for name, chunks in strategies.items()})


{'recursive': 3, 'token': 5, 'structural': 2}


<h3>두 모델을 같은 조건으로 실행</h3>
<p><strong>확인할 점</strong> 세 방식은 같은 원문에서 청크를 직접 계산합니다. 각 모델은 해당 청크와 질문 하나만 임베딩하며, 이 작은 사례만으로 최종 모델을 선택할 수 없습니다.</p>

In [4]:
def cosine(a: list[float], b: list[float]) -> float:
    return sum(x*y for x, y in zip(a, b)) / (sqrt(sum(x*x for x in a))*sqrt(sum(y*y for y in b)))
records = []
for chunking, chunks in strategies.items():
    for model in models:
        response = client.embeddings.create(model=model, input=[*chunks, query], dimensions=256)
        vectors = [item.embedding for item in response.data]
        ranking = sorted(enumerate(vectors[:-1]),
                         key=lambda x: cosine(x[1], vectors[-1]), reverse=True)
        chunk_ids = [f"{chunking}:{i:02d}" for i in range(len(chunks))]
        log = {"strategy": chunking, "model": model, "query": query,
               "chunk_ids": chunk_ids, "top_source": chunk_ids[ranking[0][0]],
               "tokens": response.usage.total_tokens}
        records.append(log)
        print(log)


{'strategy': 'recursive', 'model': 'text-embedding-3-small', 'query': '제주에는 배송이 얼마나 더 걸리나요?', 'chunk_ids': ['recursive:00', 'recursive:01', 'recursive:02'], 'top_source': 'recursive:01', 'tokens': 93}
{'strategy': 'recursive', 'model': 'text-embedding-3-large', 'query': '제주에는 배송이 얼마나 더 걸리나요?', 'chunk_ids': ['recursive:00', 'recursive:01', 'recursive:02'], 'top_source': 'recursive:01', 'tokens': 93}


{'strategy': 'token', 'model': 'text-embedding-3-small', 'query': '제주에는 배송이 얼마나 더 걸리나요?', 'chunk_ids': ['token:00', 'token:01', 'token:02', 'token:03', 'token:04'], 'top_source': 'token:01', 'tokens': 94}
{'strategy': 'token', 'model': 'text-embedding-3-large', 'query': '제주에는 배송이 얼마나 더 걸리나요?', 'chunk_ids': ['token:00', 'token:01', 'token:02', 'token:03', 'token:04'], 'top_source': 'token:01', 'tokens': 94}


{'strategy': 'structural', 'model': 'text-embedding-3-small', 'query': '제주에는 배송이 얼마나 더 걸리나요?', 'chunk_ids': ['structural:00', 'structural:01'], 'top_source': 'structural:00', 'tokens': 94}
{'strategy': 'structural', 'model': 'text-embedding-3-large', 'query': '제주에는 배송이 얼마나 더 걸리나요?', 'chunk_ids': ['structural:00', 'structural:01'], 'top_source': 'structural:00', 'tokens': 94}


<h3>최소 JSONL 실행 기록</h3>
<p><strong>확인할 점</strong> 로그는 전략, 모델, 질문, 전체 청크 ID, 최상위 출처 ID를 기록합니다. 전체 말뭉치 매니페스트와 캐시 통계는 확장 실습에서 추가합니다.</p>

In [5]:
assert len(records) == 6 and all(record["chunk_ids"] for record in records)
log_path = Path.cwd() / ".cache/ch10-notebook/08-api-runs.jsonl"
log_path.parent.mkdir(parents=True, exist_ok=True)
with log_path.open("w", encoding="utf-8") as file:
    for record in records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
saved = log_path.read_text(encoding="utf-8").splitlines()
assert len(saved) == 6
print({"jsonl": log_path.as_posix(), "records": len(saved)})


{'jsonl': '[사용자 폴더]/working/claude/projects/한국경제신문/.cache/ch10-notebook/08-api-runs.jsonl', 'records': 6}


<h3>일부러 실패해 보기</h3>
<p><strong>실패 사례</strong> 정답 문서가 없는 질문으로 최고 점수만 보고 모델 품질을 선언할 수 없습니다. 평가 정답이 비어 있으면 실험을 중단합니다.</p>

In [6]:
try:
    expected_doc = None
    assert expected_doc is not None, "정답 문서가 없어 검색 정확도를 계산할 수 없습니다"
except AssertionError as error:
    print(f"caught: {error}")


caught: 정답 문서가 없어 검색 정확도를 계산할 수 없습니다
